# Meeting policy analysis with Gemma 4 on Colab

Runs `llm.gpu.colab_policy_analyze`: the same Part 1 prompt (`prompts/policy_analysis_part_1.md`), output schema and
meta as the Gemini pipeline, with `google/gemma-4-31B-it` (or the faster `google/gemma-4-26B-A4B-it`) on this
runtime's GPU ([#511](https://github.com/getcommunityone/open-navigator/issues/511)). Input is a bundle exported on the
laptop by `python -m llm.gpu.colab_policy_export` into `crawl/policy_export/<export_date>/` (landing volume or Drive);
results go to `crawl/policy_analysis/<run_date>/`. Steps, secrets and cost:
`web_docs/docs/deployment/colab-gemma-policy-analysis.md`.

**A GPU run uses the owner's Colab compute units.** Run it only on the owner's explicit yes for this run, read the
usage before and after, and put the cost in the run's issue or PR (`AGENTS.md`, "GPU and paid compute").

Run the cells in order. Re-running cell 5 after a disconnect skips meetings already done for that run date.

In [ ]:
#@title 1. Check a GPU is attached (A100 recommended for the 31B model)
import shutil
import subprocess

if shutil.which("nvidia-smi") is None or subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise SystemExit("No GPU attached. Runtime > Change runtime type > A100 GPU > Save, then run this cell again.")
info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                      capture_output=True, text=True).stdout.strip()
print(info)
mem_gb = float(info.split(",")[-1]) / 1024
if mem_gb < 22:
    print(f"WARNING: {mem_gb:.0f} GB is too small for Gemma 4 31B even in 4-bit; use an A100 (or L4 with the 26B model).")

In [ ]:
#@title 2. Get the code (packages/llm) and install transformers + bitsandbytes
import os
import subprocess
import sys

branch = "main"  #@param {type:"string"}
REPO = "/content/open-navigator"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", branch,
                    "https://github.com/getcommunityone/open-navigator.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.5.0", "accelerate>=1.0",
                "bitsandbytes>=0.45", "databricks-sdk", "loguru", "pydantic>=2", "pyyaml"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", f"{REPO}/packages/llm"], check=True)
sys.path.insert(0, f"{REPO}/packages/llm/src")
os.environ.setdefault("LLM_USAGE_LOG_DIR", "/content/llm_usage")

import transformers  # noqa: E402
from llm.gpu import colab_io  # noqa: E402
from llm.gpu import colab_policy_analyze as cpa  # noqa: E402
print("transformers", transformers.__version__, "| llm.gpu.colab_policy_analyze ready")
print("If transformers was upgraded and the import fails: Runtime > Restart session, then run cells 2-3 again.")

In [ ]:
#@title 3. Read the secrets (values are never printed)
# Colab: the key icon in the left bar > add HF_TOKEN (and DATABRICKS_HOST, DATABRICKS_TOKEN) > allow this notebook.
present = {name: bool(colab_io.get_secret(name)) for name in cpa.SECRET_NAMES}
hf = colab_io.get_secret("HF_TOKEN")
if hf:
    os.environ["HF_TOKEN"] = hf   # the Gemma 4 weights are gated: accept the licence on the model page first
print({k: ("set" if v else "missing") for k, v in present.items()})
if not present["HF_TOKEN"]:
    raise SystemExit("Add the HF_TOKEN secret (a read token whose account accepted the Gemma licence), then rerun.")
if not (present["DATABRICKS_HOST"] and present["DATABRICKS_TOKEN"]):
    if os.path.isdir(colab_io.DRIVE_ROOT + "/policy_export"):
        print("Bundle already on this runtime; skipping the Drive mount.")   # no popup needed
    else:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(colab_io.DRIVE_ROOT, exist_ok=True)
    print("No Databricks secrets: using the Drive folder", colab_io.DRIVE_ROOT,
          "(the bundle goes under policy_export/<export_date>/ there)")

In [ ]:
#@title 4. Usage before
#@markdown Read the compute-unit balance in Colab's **Resources** panel (top right, RAM/Disk > View resources) and
#@markdown the rate it shows for this GPU, and type them here.
cu_balance_before = 0.0  #@param {type:"number"}
cu_per_hour = 0.0  #@param {type:"number"}
from datetime import datetime, timezone  # noqa: E402

from llm.usage.gpu import gpu_names  # noqa: E402

GPU = colab_io.gpu_kind((gpu_names() or [None])[0])
SESSION_STARTED = datetime.now(timezone.utc).isoformat()
print(f"usage before: {SESSION_STARTED[:19]}Z gpu={GPU} cu_balance={cu_balance_before or 'not entered'} "
      f"cu_per_hour={cu_per_hour or colab_io.COLAB_CU_PER_HOUR.get(GPU)}")

In [ ]:
#@title 5. Run
#@markdown `pilot` runs the meetings in `pilot_videos.txt` (they already have a Gemini analysis) and writes a
#@markdown comparison; `full` runs `videos.txt`.
#@markdown Leave `export_date` blank to use the newest bundle under `policy_export/`.
export_date = ""  #@param {type:"string"}
mode = "pilot"  #@param ["pilot", "full"]
pilot_meetings = 3  #@param {type:"integer"}
model = "google/gemma-4-31B-it"  #@param ["google/gemma-4-31B-it", "google/gemma-4-26B-A4B-it"]
quantize = "auto"  #@param ["auto", "4bit", "none"]
limit = 200  #@param {type:"integer"}
#@markdown Leave blank for today (UTC). To resume a run from an earlier day, enter that day.
run_date = ""  #@param {type:"string"}
owner_said_yes_to_this_run = False  #@param {type:"boolean"}

if not owner_said_yes_to_this_run:
    raise SystemExit("A GPU run needs the owner's explicit yes for this run. Tick the box once you have it.")
store = colab_io.make_store()
export_date = export_date.strip() or colab_io.latest_child(store, "policy_export")
if not export_date:
    raise SystemExit("No bundle under policy_export/ in " + store.label + ". Export one on the laptop first.")
print("bundle:", export_date)
RUN_DATE = run_date.strip() or colab_io.today_utc()
argv = ["--videos-file", f"policy_export/{export_date}/videos.txt", "--model", model, "--quantize", quantize,
        "--limit", str(limit), "--run-date", RUN_DATE]
if mode == "pilot":
    argv += ["--pilot", str(pilot_meetings)]
if cu_per_hour:
    argv += ["--cu-per-hour", str(cu_per_hour)]
exit_code = cpa.main(argv)
print("results in", f"crawl/{cpa.DEFAULT_OUT}/{RUN_DATE}/", "| exit code", exit_code)

In [ ]:
#@title 6. Usage after (record the cost)
cu_balance_after = 0.0  #@param {type:"number"}
store = colab_io.make_store()
row = cpa.record_measured_cu(store, cpa.DEFAULT_OUT, RUN_DATE, cu_balance_before or None, cu_balance_after or None,
                             GPU, started_after=SESSION_STARTED)
measured = (f"{row['compute_units']} CU measured, apportioned to {row['meeting_count']} meeting(s) by GPU seconds"
            if row else "no measured CU (enter both balances)")
print(f"usage after: {datetime.now(timezone.utc).isoformat(timespec='seconds')} gpu={GPU} "
      f"cu_balance={cu_balance_after or 'not entered'}; {measured}")

In [ ]:
#@title 7. Summary
run_rel = f"{cpa.DEFAULT_OUT}/{RUN_DATE}"
rows = [r for r in cpa.read_jsonl(store, f"{run_rel}/usage.jsonl") if r.get("row_kind") == "meeting"]
print(cpa.summary_table(rows))
print("\nPut the usage lines above and this table in the run's issue or PR.")
if mode == "pilot":
    print(f"\nPilot comparison: crawl/{run_rel}/comparison. Copy that folder to the laptop and score it there:")
    print("  python -m llm.gemini.compare_models --rescore <folder>/comparison")

In [ ]:
#@title 8. Finish: keep the results, then release the runtime
#@markdown An idle GPU runtime keeps using compute units, and Colab can recycle it and wipe its disk. This releases the
#@markdown runtime **only when the results are somewhere that survives it**: the landing volume, or a mounted Drive.
release_runtime = True  #@param {type:"boolean"}
#@markdown Tick this only if you already copied the run folder off this runtime by hand.
results_copied_elsewhere = False  #@param {type:"boolean"}

on_volume = isinstance(store, colab_io.DatabricksVolumeStore)
on_drive = os.path.ismount("/content/drive")
safe = on_volume or on_drive or results_copied_elsewhere
where = "the landing volume" if on_volume else ("Google Drive" if on_drive else "THIS runtime's disk only")
print("results are on:", where, "|", store.label)
if release_runtime:
    if not safe:
        raise SystemExit("Not releasing: the results exist only on this runtime's disk, and releasing wipes it. "
                         "Copy the run folder off first, or tick results_copied_elsewhere.")
    from google.colab import runtime
    print("releasing the runtime")
    runtime.unassign()
else:
    print("Left running. Remember: it keeps using compute units until you disconnect it.")